<a href="https://colab.research.google.com/github/Nishikant090/Proactive-Handover-Trigger-Prediction-in-5G-Networks-Using-Time-Series-Forecasting-Models/blob/main/MULTIFEATURE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# =============================
# 1️⃣ Load dataset
# =============================
import pandas as pd
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

# =============================
# 2️⃣ Select Features
# =============================
features = ['RSRP','RSRQ','CINR','PCI']
data = df[features].dropna()

# =============================
# 3️⃣ Normalize
# =============================
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaled = scaler.fit_transform(data)

# =============================
# 4️⃣ Create Sequences
# =============================
import numpy as np

TIMESTEPS = 10

X = []
y = []

for i in range(TIMESTEPS, len(scaled)):
    X.append(scaled[i-TIMESTEPS:i])
    y.append(scaled[i,0])   # predicting RSRP

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


Saving drive_test_measurements01.csv to drive_test_measurements01 (2).csv
Saving drive_test_measurements02.csv to drive_test_measurements02 (2).csv
Saving drive_test_measurements03.csv to drive_test_measurements03 (2).csv
X shape: (1708, 10, 4)
y shape: (1708,)


In [12]:
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

def transformer_block(x):
    attn = MultiHeadAttention(key_dim=32, num_heads=2)(x, x)
    x = Add()([x, attn])
    x = LayerNormalization()(x)

    ff = Dense(64, activation="relu")(x)
    ff = Dense(x.shape[-1])(ff)

    x = Add()([x, ff])
    x = LayerNormalization()(x)
    return x

inputs = Input(shape=(TIMESTEPS, 4))

x = GRU(64, return_sequences=True)(inputs)
x = transformer_block(x)
x = LSTM(64, return_sequences=True)(x)

x = GlobalAveragePooling1D()(x)
outputs = Dense(1)(x)

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='mse')

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10, 4)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ (None, 10, 64)    │     13,440 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 64)    │     16,640 │ gru[0][0],        │
│ (MultiHeadAttentio… │                   │            │ gru[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 64)    │          0 │ gru[0][0],        │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 10, 64)    │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 10, 64)    │      4,160 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 10, 64)    │      4,160 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 10, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 10, 64)    │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ lstm[0][0]        │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         65 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 71,745 (280.25 KB)

 Trainable params: 71,745 (280.25 KB)

 Non-trainable params: 0 (0.00 B)